In [17]:

# Download the shared project code from GitHub
!wget -O sepsis.py https://raw.githubusercontent.com/kna20/early-sepsis-prediction/main/sepsis.py

# Import Bahiya's existing functions
import sepsis as s
import pandas as pd
import numpy as np

# Verify that everything loaded correctly
print("sepsis.py imported successfully!")

print("Vital signs:", len(s.VITALS))
print("Lab features:", len(s.LABS))
print("Total original features:", len(s.ALL_FEATURES))


--2026-11-15 13:52:56--  https://raw.githubusercontent.com/kna20/early-sepsis-prediction/main/sepsis.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15228 (15K) [text/plain]
Saving to: ‘sepsis.py’

sepsis.py           100%[===================>]  14.87K  --.-KB/s    in 0.001s  

2026-11-15 13:52:56 (21.0 MB/s) - ‘sepsis.py’ saved [15228/15228]

sepsis.py imported successfully!
Vital signs: 8
Lab features: 26
Total original features: 40


In [18]:

def add_trends(data, features, window=3):
    df = data.sort_values(["PatientID", "Hour"]).copy()

    for feature in features:
        # Forward-fill within each patient
        filled = df.groupby("PatientID")[feature].ffill()
        grouped = filled.groupby(df["PatientID"])

        # Rolling mean
        df[f"{feature}_mean_{window}h"] = grouped.transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        ).astype("float32")

        # Rolling standard deviation
        df[f"{feature}_std_{window}h"] = grouped.transform(
            lambda x: x.rolling(window, min_periods=1).std(ddof=0)
        ).astype("float32")

        # Change from window hours earlier
        df[f"{feature}_change_{window}h"] = (
            filled - grouped.shift(window)
        ).astype("float32")

    return df


In [19]:

# Load the real Hospital A dataset
data_A = s.load_hospital("A")

# Use Bahiya's exact 80/20 patient split
dev_ids, test_ids = s.split_patients(data_A)

print("Total patients:", data_A["PatientID"].nunique())
print("Total hourly observations:", len(data_A))
print("Development patients:", len(dev_ids))
print("Held-out test patients:", len(test_ids))
print("Patient overlap:", len(set(dev_ids) & set(test_ids)))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total patients: 20336
Total hourly observations: 790215
Development patients: 16268
Held-out test patients: 4068
Patient overlap: 0


In [20]:

# Create the same 5 patient-level folds as Bahiya
fold_of = s.make_folds(data_A, dev_ids)

# Prepare the development data using her existing pipeline
dev = s.prepare_dev(data_A, dev_ids, fold_of)

print("Development observations:", len(dev["labels"]))
print("Number of CV folds:", len(np.unique(dev["fold"])))
print("Original feature columns:", dev["filled"].shape[1])
print("Missingness flags:", dev["flags"].shape[1])
print("Hours-since features:", dev["since"].shape[1])


Development observations: 631120
Number of CV folds: 5
Original feature columns: 40
Missingness flags: 34
Hours-since features: 34


In [21]:

# Select vital signs for our first experiment
vital_features = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"
]

# Get the already forward-filled development data
trend_input = dev["filled"][vital_features].copy()

# Add patient identifiers and hours
trend_input["PatientID"] = dev["patients"]
trend_input["Hour"] = dev["hours"]

# Apply the function we created earlier
trend_data = add_trends(
    trend_input,
    features=vital_features,
    window=3
)

# Identify the newly created columns
trend_cols = [
    f"{feature}_{stat}_3h"
    for feature in vital_features
    for stat in ["mean", "std", "change"]
]

# Verify the rows still match Bahiya's prepared data
assert np.array_equal(
    trend_data["PatientID"].to_numpy(), dev["patients"]
)
assert np.array_equal(
    trend_data["Hour"].to_numpy(), dev["hours"]
)

print("Development observations:", len(trend_data))
print("Number of new trend features:", len(trend_cols))
print("Trend features:", trend_cols)

# Inspect the first patient's heart-rate trends
display(
    trend_data[[
        "PatientID", "Hour", "HR",
        "HR_mean_3h", "HR_std_3h", "HR_change_3h"
    ]].head(10).round(2)
)


Development observations: 631120
Number of new trend features: 21
Trend features: ['HR_mean_3h', 'HR_std_3h', 'HR_change_3h', 'O2Sat_mean_3h', 'O2Sat_std_3h', 'O2Sat_change_3h', 'Temp_mean_3h', 'Temp_std_3h', 'Temp_change_3h', 'SBP_mean_3h', 'SBP_std_3h', 'SBP_change_3h', 'MAP_mean_3h', 'MAP_std_3h', 'MAP_change_3h', 'DBP_mean_3h', 'DBP_std_3h', 'DBP_change_3h', 'Resp_mean_3h', 'Resp_std_3h', 'Resp_change_3h']


,PatientID,Hour,HR,HR_mean_3h,HR_std_3h,HR_change_3h
0,p000001,0,NaN,NaN,NaN,NaN
1,p000001,1,97.0,97.0,0.00,NaN
2,p000001,2,89.0,93.0,4.00,NaN
3,p000001,3,90.0,92.0,3.56,NaN
4,p000001,4,103.0,94.0,6.38,6.0
5,p000001,5,110.0,101.0,8.29,21.0
6,p000001,6,108.0,107.0,2.94,18.0
7,p000001,7,106.0,108.0,1.63,3.0
8,p000001,8,104.0,106.0,1.63,-6.0
9,p000001,9,102.0,104.0,1.63,-6.0


In [22]:

# Register the 21 trend features in Bahiya's prepared data
dev["filled"][trend_cols] = trend_data[trend_cols].to_numpy()

# Record which trend values are missing
dev["missing"][trend_cols] = (
    trend_data[trend_cols].isna().to_numpy()
)

# Define the features for our first experiment
baseline_cols = s.ALL_FEATURES
trend_model_cols = s.ALL_FEATURES + trend_cols

# Check everything
print("Original features:", len(baseline_cols))
print("New trend features:", len(trend_cols))
print("Total features before sparse-column removal:", len(trend_model_cols))

print("Trends registered:", all(
    col in dev["filled"].columns and col in dev["missing"].columns
    for col in trend_cols
))

print("Missing trend values:",
      int(dev["filled"][trend_cols].isna().sum().sum()))


Original features: 40
New trend features: 21
Total features before sparse-column removal: 61
Trends registered: True
Missing trend values: 1401409


In [23]:

# Experiment 1: Original features + 3-hour vital trends
oof_trends = s.run_cv(
    dev,
    trend_model_cols,
    class_weight="balanced"
)

# Calculate AUROC and AUPRC across the five folds
trend_results = {"Vitals trends (3h)": oof_trends}

auc_summary, _ = s.compare_auc(trend_results)

print("AUROC and AUPRC:")
display(auc_summary.round(4))

# Find the threshold with the highest clinical utility
cutoff_results = s.best_cutoff_table(trend_results)

print("Best threshold and clinical metrics:")
display(cutoff_results.round(4))


trained on 57 input features
trained on 57 input features
trained on 57 input features
trained on 57 input features
trained on 57 input features
AUROC and AUPRC:


AUROC          AUPRC        
                      mean     std   mean     std
Features                                         
Vitals trends (3h)  0.7531  0.0229  0.084  0.0148

Best threshold and clinical metrics:


,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
Vitals trends (3h),0.55,0.1098,0.061,0.553,0.1892,0.3159,0.3158,0.0421


In [24]:

# All 26 laboratory features
lab_features = list(s.LABS)

# Get forward-filled laboratory measurements
lab_input = dev["filled"][lab_features].copy()

# Add patient IDs and hours
lab_input["PatientID"] = dev["patients"]
lab_input["Hour"] = dev["hours"]

# Generate 3-hour laboratory trends
lab_trend_data = add_trends(
    lab_input,
    features=lab_features,
    window=3
)

# Identify the 78 new laboratory trend columns
lab_trend_cols = [
    f"{feature}_{stat}_3h"
    for feature in lab_features
    for stat in ["mean", "std", "change"]
]

# Register them in Bahiya's pipeline
# Check that patient-hour rows are aligned
assert np.array_equal(
    lab_trend_data["PatientID"].to_numpy(), dev["patients"]
)
assert np.array_equal(
    lab_trend_data["Hour"].to_numpy(), dev["hours"]
)

# Add laboratory trends together to avoid fragmentation warnings
lab_values = lab_trend_data[lab_trend_cols].reset_index(drop=True)

dev["filled"] = pd.concat(
    [dev["filled"], lab_values], axis=1
).copy()

dev["missing"] = pd.concat(
    [dev["missing"], lab_values.isna()], axis=1
).copy()

# Combine original, vital-trend and lab-trend features
vital_lab_cols = list(s.ALL_FEATURES) + trend_cols + lab_trend_cols

print("Original features:", len(s.ALL_FEATURES))
print("Vital trend features:", len(trend_cols))
print("Lab trend features:", len(lab_trend_cols))
print("Total features:", len(vital_lab_cols))
print("All trends registered:", all(
    col in dev["filled"].columns and col in dev["missing"].columns
    for col in vital_lab_cols
))


Original features: 40
Vital trend features: 21
Lab trend features: 78
Total features: 139
All trends registered: True


In [25]:

# Experiment 2: Original features + vital and lab trends
oof_vital_lab = s.run_cv(
    dev,
    vital_lab_cols,
    class_weight="balanced"
)

# Compare Experiment 1 and Experiment 2
results_compare = {
    "Vitals trends (3h)": oof_trends,
    "Vitals + lab trends (3h)": oof_vital_lab
}

# AUROC and AUPRC
auc_summary, auc_differences = s.compare_auc(results_compare)

print("AUROC and AUPRC:")
display(auc_summary.round(4))

print("Fold-by-fold differences:")
display(auc_differences.round(4))

# Clinical utility and other metrics
cutoff_results = s.best_cutoff_table(results_compare)

print("Clinical utility and other metrics:")
display(cutoff_results.round(4))


trained on 135 input features
trained on 135 input features
trained on 135 input features
trained on 135 input features
trained on 135 input features
AUROC and AUPRC:


AUROC           AUPRC        
                            mean     std    mean     std
Features                                                
Vitals trends (3h)        0.7531  0.0229  0.0840  0.0148
Vitals + lab trends (3h)  0.7579  0.0217  0.0846  0.0149

Fold-by-fold differences:


,AUROC,AUPRC
Features,Vitals + lab trends (3h),Vitals + lab trends (3h)
1,0.0043,0.0013
2,0.0085,0.0011
3,0.0043,-0.0003
4,0.0028,-0.0007
5,0.0039,0.0016
mean,0.0048,0.0006
std,0.0022,0.0010


Clinical utility and other metrics:


,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
Vitals trends (3h),0.55,0.1098,0.0610,0.5530,0.1892,0.3159,0.3158,0.0421
Vitals + lab trends (3h),0.55,0.1119,0.0622,0.5597,0.1874,0.3251,0.3251,0.0377


In [26]:

# Experiment 3: Vital trends + missingness features
oof_vital_missing = s.run_cv(
    dev,
    trend_model_cols,
    flags=True,
    since=True,
    class_weight="balanced"
)

# Compare with Experiment 1
results_compare = {
    "Vitals trends (3h)": oof_trends,
    "Vitals trends + missingness": oof_vital_missing
}

auc_summary, auc_differences = s.compare_auc(results_compare)

print("AUROC and AUPRC:")
display(auc_summary.round(4))

print("Fold-by-fold differences:")
display(auc_differences.round(4))

print("Clinical utility and other metrics:")
cutoff_results = s.best_cutoff_table(results_compare)
display(cutoff_results.round(4))


trained on 117 input features
trained on 117 input features
trained on 117 input features
trained on 117 input features
trained on 117 input features
AUROC and AUPRC:


AUROC           AUPRC        
                               mean     std    mean     std
Features                                                   
Vitals trends (3h)           0.7531  0.0229  0.0840  0.0148
Vitals trends + missingness  0.7883  0.0236  0.0948  0.0168

Fold-by-fold differences:


,AUROC,AUPRC
Features,Vitals trends + missingness,Vitals trends + missingness
1,0.0359,0.0129
2,0.0313,0.0114
3,0.0375,0.0072
4,0.0338,0.0131
5,0.0376,0.0091
mean,0.0352,0.0107
std,0.0027,0.0026


Clinical utility and other metrics:


,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
Vitals trends (3h),0.55,0.1098,0.0610,0.5530,0.1892,0.3159,0.3158,0.0421
Vitals trends + missingness,0.55,0.1188,0.0657,0.6193,0.1955,0.3715,0.3714,0.0430


In [27]:

# Experiment 4: Vital + lab trends + missingness
oof_all_trends_missing = s.run_cv(
    dev,
    vital_lab_cols,
    flags=True,
    since=True,
    class_weight="balanced"
)

# Compare Experiment 3 and Experiment 4
results_compare = {
    "Vitals trends + missingness": oof_vital_missing,
    "Vitals + lab trends + missingness": oof_all_trends_missing
}

# AUROC and AUPRC
auc_summary, auc_differences = s.compare_auc(results_compare)

print("AUROC and AUPRC:")
display(auc_summary.round(4))

print("Fold-by-fold differences:")
display(auc_differences.round(4))

# Clinical utility and other metrics
cutoff_results = s.best_cutoff_table(results_compare)

print("Clinical utility and other metrics:")
display(cutoff_results.round(4))


trained on 195 input features
trained on 195 input features
trained on 195 input features
trained on 195 input features
trained on 195 input features
AUROC and AUPRC:


AUROC           AUPRC        
                                     mean     std    mean     std
Features                                                         
Vitals trends + missingness        0.7883  0.0236  0.0948  0.0168
Vitals + lab trends + missingness  0.7873  0.0241  0.0948  0.0175

Fold-by-fold differences:


,AUROC,AUPRC
Features,Vitals + lab trends + missingness,Vitals + lab trends + missingness
1,-0.0004,0.0008
2,-0.0015,-0.0016
3,-0.0020,-0.0005
4,-0.0006,0.0009
5,-0.0008,0.0005
mean,-0.0011,0.0000
std,0.0007,0.0010


Clinical utility and other metrics:


,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
Vitals trends + missingness,0.55,0.1188,0.0657,0.6193,0.1955,0.3715,0.3714,0.0430
Vitals + lab trends + missingness,0.55,0.1192,0.0660,0.6150,0.1932,0.3698,0.3697,0.0426
